# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata (as an object)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The `mlcroissant` Dataset object provides access to record sets and their corresponding fields. Every entity is referenced by its `@id`.

In [ ]:
# List all available record sets and their @id
record_sets = dataset.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f"- {rs['@id']} : {rs['name']}")

# For demonstration, print all fields of each record set
overview = {}
for rs in record_sets:
    fields = rs.get('fields', [])
    field_ids = [f["@id"] for f in fields]
    print(f"\nRecord Set @id: {rs['@id']}")
    for f in fields:
        print(f"    Field @id: {f['@id']} | name: {f['name']} | type: {f.get('dataType', '')}")
    overview[rs['@id']] = field_ids

## 3. Data Extraction
Load data from each available record set.
Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare to extract all record sets, referencing them by @id
rs_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns from first record set
if rs_ids:
    print(f"Available columns in record set {rs_ids[0]}:")
    print(dataframes[rs_ids[0]].columns.tolist())
    dataframes[rs_ids[0]].head()
else:
    print("No record sets found in this dataset package.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to prepare for analysis.
- Filtering records based on criteria
- Normalizing numeric fields
- Grouping by key attributes

For demonstration, select a numeric field (e.g., `Age`) and a grouping field (e.g., `Sex`) from the fields overview, referencing by their `@id`.

In [ ]:
# Choose record set and fields relevant for EDA

# For this dataset: most clinical variables are in the main table.
# We'll select the first available record set.
record_set_id = rs_ids[0] if rs_ids else None  # fallback

# Find the field @id for 'Age' and 'Sex'
age_field_id = None
sex_field_id = None

fields = record_sets[0]['fields'] if record_sets else []
for f in fields:
    if f['name'].lower() == 'age':
        age_field_id = f['@id']
    elif f['name'].lower() == 'sex':
        sex_field_id = f['@id']

if record_set_id and age_field_id:
    df = dataframes[record_set_id]
    # Drop rows with missing Age
    if age_field_id in df:
        # Filter for Age > 50
        threshold = 50
        filtered_df = df[df[age_field_id] > threshold].copy()
        print(f"Filtered records with {age_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize Age
        filtered_df[f"{age_field_id}_normalized"] = (
            filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
        print(f"Normalized {age_field_id} for filtered records:")
        print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

        # Group by 'Sex' field, if available
        if sex_field_id and sex_field_id in df:
            grouped_df = filtered_df.groupby(sex_field_id).mean(numeric_only=True)
            print(f"Grouped data by {sex_field_id}:")
            print(grouped_df[[age_field_id, f"{age_field_id}_normalized"]].head())
    else:
        print(f"Field {age_field_id} not found in DataFrame.")
else:
    print("Cannot perform EDA: 'Age' field or record set not found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib.

In [ ]:
# Simple histogram of normalized Age
if record_set_id and age_field_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    if age_field_id in df:
        plt.figure(figsize=(7, 4))
        df[age_field_id].hist(bins=10, color="skyblue")
        plt.title("Age distribution")
        plt.xlabel("Age")
        plt.ylabel("Count")
        plt.show()
        if sex_field_id and sex_field_id in df:
            plt.figure(figsize=(7, 4))
            for sex_label, group in df.groupby(sex_field_id):
                group[age_field_id].hist(alpha=0.5, bins=10, label=f'{sex_label}')
            plt.title("Age distribution by Sex")
            plt.xlabel("Age")
            plt.ylabel("Count")
            plt.legend()
            plt.show()
else:
    print("Visualization fields missing.")

## 6. Conclusion
This notebook demonstrated how to load, overview, extract, and analyze the FAIR² dataset using `mlcroissant`, referencing all entities by their `@id`. Key steps included extracting and visualizing demographic variables (e.g., Age, Sex). The dataset is suitable for clinical stratification studies and biomarker analysis in colorectal cancer survivors.

For further exploration, consult the dataset documentation at:
- https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
- mlcroissant library documentation: https://github.com/mlcommons/croissant